# FT-Code training on Colab T4

Trains FT-Code-300, FT-Code-1000, FT-Code-5000 from CodeSearchNet/python on a T4 runtime. Mounts Drive at `/content/drive/MyDrive/adaptmem-bench/ft-code/` to persist checkpoints.

Expected runtime on T4: ~5 min per FT-Code-300, ~15 min per FT-Code-1000, ~75 min per FT-Code-5000. Total ~95 min including data download + indexing.

Verify GPU at the top, run end-to-end. Each train cell saves its own checkpoint, so a runtime interruption only loses the cell in flight.

In [1]:
# 1. Sanity: confirm T4 GPU is attached
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Switch runtime to GPU (Runtime > Change runtime type > T4 GPU)'

cuda available: True
device: Tesla T4


In [2]:
# 2. Install dependencies and clone the repo (public)
!pip install -q sentence-transformers datasets numpy
!git clone --depth 1 https://github.com/nakata-app/adaptmem.git /content/adaptmem
%cd /content/adaptmem
!pip install -q -e .

Cloning into '/content/adaptmem'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 114 (delta 14), reused 83 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 5.05 MiB | 11.10 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/adaptmem
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adaptmem (pyproject.toml) ... done


In [8]:
# 3. Mount Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/adaptmem-bench/ft-code'
os.makedirs(OUT_DIR, exist_ok=True)
print('checkpoints will land in', OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoints will land in /content/drive/MyDrive/adaptmem-bench/ft-code


In [4]:
# 4. Run the training script for all three sizes
import os, sys, json, time
from pathlib import Path
sys.path.insert(0, '/content/adaptmem/benchmarks')
from codesearchnet_train_colab import train_one

os.environ['ADAPTMEM_DEVICE'] = 'cuda'
out = Path(OUT_DIR)
results = []
for n in [300, 1000, 5000]:
    print(f'\n--- starting FT-Code-{n} ---')
    t0 = time.time()
    r = train_one(n, out, device='cuda')
    print(f'FT-Code-{n} done in {time.time()-t0:.1f}s')
    results.append(r)

summary_path = out / 'training_summary.json'
summary_path.write_text(json.dumps(results, indent=2, default=str))
print(f'\nALL DONE → {summary_path}')


--- starting FT-Code-300 ---

=== FT-Code-300 ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/28.7M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

{'loss': '0.1816', 'grad_norm': '0.4506', 'learning_rate': '4.476e-06', 'epoch': '0.8'}
{'train_runtime': '93.06', 'train_samples_per_second': '53.73', 'train_steps_per_second': '6.716', 'train_loss': '0.1613', 'epoch': '1'}
  train: {'n_pairs': 5000, 'runtime_s': 95.48, 'n_steps': 625, 'n_tokens_approx': 1398182, 'tokens_per_s': 14644.4}, 178.0s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved → /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-5000
FT-Code-5000 done in 180.9s

ALL DONE → /content/drive/MyDrive/adaptmem-bench/ft-code/training_summary.json


In [6]:
!cd /content/adaptmem && git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.11 KiB | 227.00 KiB/s, done.
From https://github.com/nakata-app/adaptmem
   42ae528..f67c55a  master     -> origin/master
Updating 42ae528..f67c55a
Fast-forward
 benchmarks/codesearchnet_eval.py | 11 ++++++-----
 1 file changed, 6 insertions(+), 5 deletions(-)


In [7]:
!git clone --depth 1 https://github.com/nakata-app/adaptmem.git /content/adaptmem 2>/dev/null

%cd /content/adaptmem

!git pull

!pip install -q -e .

import sys
sys.path.insert(0, '/content/adaptmem/benchmarks')

print('READY')

/content/adaptmem
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adaptmem (pyproject.toml) ... done
READY


In [9]:
# 5. Optional: quick eval on a 1000-query test sample for each checkpoint
from codesearchnet_eval import evaluate
from pathlib import Path

for n in [300, 1000, 5000]:
    ckpt = Path(OUT_DIR) / f'ft-code-{n}'
    res_path = Path(OUT_DIR) / f'eval_ft-code-{n}_n1000.jsonl'
    print(f'\n--- evaluating FT-Code-{n} on 1000 test queries ---')
    evaluate(ckpt, n=1000, out=res_path)


--- evaluating FT-Code-300 on 1000 test queries ---
[eval] loading model from /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-300


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[eval] loading test split (n=1000)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[eval] 1000 queries, 1000 corpus entries in 2.7s
[eval] encoding test corpus with checkpoint model
[eval] encoded in 3.0s
[eval] R@1=0.8390 R@5=0.9600 R@10=0.9740 MRR=0.8938
[eval] per-query → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-300_n1000.jsonl
[eval] summary  → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-300_n1000.summary.json

--- evaluating FT-Code-1000 on 1000 test queries ---
[eval] loading model from /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-1000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[eval] loading test split (n=1000)
[eval] 1000 queries, 1000 corpus entries in 1.4s
[eval] encoding test corpus with checkpoint model
[eval] encoded in 2.7s
[eval] R@1=0.9040 R@5=0.9810 R@10=0.9950 MRR=0.9389
[eval] per-query → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-1000_n1000.jsonl
[eval] summary  → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-1000_n1000.summary.json

--- evaluating FT-Code-5000 on 1000 test queries ---
[eval] loading model from /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-5000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[eval] loading test split (n=1000)
[eval] 1000 queries, 1000 corpus entries in 1.4s
[eval] encoding test corpus with checkpoint model
[eval] encoded in 3.1s
[eval] R@1=0.9160 R@5=0.9870 R@10=0.9960 MRR=0.9484
[eval] per-query → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-5000_n1000.jsonl
[eval] summary  → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-5000_n1000.summary.json


In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
from datasets import load_dataset
import sys

sys.path.insert(0, '/content/adaptmem/benchmarks')
from codesearchnet_eval import load_test

print("[baseline] loading default all-MiniLM-L6-v2 (no FT)")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

queries, truths, corpus_ids = load_test(1000)

ds = load_dataset("code_search_net", "python", split="test")

body_by_id = {
    f"ts{i}": row.get("func_code_string")
    for i, row in enumerate(ds)
}

corpus = [body_by_id[cid] for cid in corpus_ids]

query_emb = model.encode(
    queries,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

corpus_emb = model.encode(
    corpus,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

sims = query_emb @ corpus_emb.T
ranks = np.argsort(-sims, axis=1)

cid_to_idx = {c: i for i, c in enumerate(corpus_ids)}

r1 = r5 = r10 = 0
mrr = 0.0

for qi, t in enumerate(truths):
    pos = np.where(ranks[qi] == cid_to_idx[t])[0][0]

    if pos == 0:
        r1 += 1
    if pos < 5:
        r5 += 1
    if pos < 10:
        r10 += 1

    mrr += 1.0 / (pos + 1)

n = len(queries)

print(
    f"[baseline] R@1={r1/n:.4f} "
    f"R@5={r5/n:.4f} "
    f"R@10={r10/n:.4f} "
    f"MRR={mrr/n:.4f}"
)

[baseline] loading default all-MiniLM-L6-v2 (no FT)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[baseline] R@1=0.7540 R@5=0.9240 R@10=0.9420 MRR=0.8295


In [15]:
queries, truths, corpus_ids = load_test(-1)

ds2 = load_dataset("code_search_net", "python", split="test")

body_by_id = {
    f"ts{i}": (row.get("func_code_string") or "")
    for i, row in enumerate(ds2)
}

texts = [body_by_id[c] for c in corpus_ids]

corpus_emb = model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

query_emb = model.encode(
    queries,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

sims = query_emb @ corpus_emb.T
ranks = np.argsort(-sims, axis=1)

cid_to_idx = {c: i for i, c in enumerate(corpus_ids)}

r1 = r5 = r10 = 0
mrr = 0.0

for qi, t in enumerate(truths):
    pos = int(np.where(ranks[qi] == cid_to_idx[t])[0][0])

    if pos == 0:
        r1 += 1
    if pos < 5:
        r5 += 1
    if pos < 10:
        r10 += 1

    mrr += 1.0 / (pos + 1)

n = len(queries)

print(
    f"[baseline-full] R@1={r1/n:.4f} R@5={r5/n:.4f} "
    f"R@10={r10/n:.4f} MRR={mrr/n:.4f}"
)

[baseline-full] R@1=0.6477 R@5=0.8551 R@10=0.8972 MRR=0.7406


In [18]:
from googleapiclient.discovery import build
from google.colab import auth

auth.authenticate_user()

drive = build("drive", "v3")

for name in ["ft-code-300", "ft-code-1000", "ft-code-5000"]:
    res = drive.files().list(
        q=f"name='{name}' and mimeType='application/vnd.google-apps.folder'",
        fields="files(id,name)"
    ).execute()

    fid = res["files"][0]["id"]

    drive.permissions().create(
        fileId=fid,
        body={"role": "reader", "type": "anyone"}
    ).execute()

    print(name, f"https://drive.google.com/drive/folders/{fid}")

ft-code-300 https://drive.google.com/drive/folders/1fe5t5LWWHFGDV5CC5GcfCk5aOVaRKKRb
ft-code-1000 https://drive.google.com/drive/folders/1QhjDc63M4vKdOxVMP7ZbpyhYoLcXjnlC
ft-code-5000 https://drive.google.com/drive/folders/1GZXGQG4LJL8jm1ajbo8JgiIPPICf_QtT


In [17]:
from googleapiclient.discovery import build
from google.colab import auth

auth.authenticate_user()

drive = build("drive", "v3")

res = drive.files().list(
    q="name='ft-code-5000' and mimeType='application/vnd.google-apps.folder'",
    fields="files(id,name)"
).execute()

fid = res["files"][0]["id"]

drive.permissions().create(
    fileId=fid,
    body={"role": "reader", "type": "anyone"}
).execute()

print(f"https://drive.google.com/drive/folders/{fid}")

https://drive.google.com/drive/folders/1GZXGQG4LJL8jm1ajbo8JgiIPPICf_QtT


In [14]:
# 6. Optional: full 19k-query eval for the final report. Slower; ~3-5 min per checkpoint on T4.
for n in [300, 1000, 5000]:
    ckpt = Path(OUT_DIR) / f'ft-code-{n}'
    res_path = Path(OUT_DIR) / f'eval_ft-code-{n}_full.jsonl'
    print(f'\n--- evaluating FT-Code-{n} on full test split ---')
    evaluate(ckpt, n=-1, out=res_path)


--- evaluating FT-Code-300 on full test split ---
[eval] loading model from /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-300


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[eval] loading test split (n=all)
[eval] 21935 queries, 21935 corpus entries in 5.8s
[eval] encoding test corpus with checkpoint model
[eval] encoded in 59.4s
[eval] R@1=0.8004 R@5=0.9406 R@10=0.9585 MRR=0.8636
[eval] per-query → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-300_full.jsonl
[eval] summary  → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-300_full.summary.json

--- evaluating FT-Code-1000 on full test split ---
[eval] loading model from /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-1000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[eval] loading test split (n=all)
[eval] 21935 queries, 21935 corpus entries in 5.8s
[eval] encoding test corpus with checkpoint model
[eval] encoded in 61.6s
[eval] R@1=0.9020 R@5=0.9760 R@10=0.9807 MRR=0.9360
[eval] per-query → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-1000_full.jsonl
[eval] summary  → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-1000_full.summary.json

--- evaluating FT-Code-5000 on full test split ---
[eval] loading model from /content/drive/MyDrive/adaptmem-bench/ft-code/ft-code-5000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[eval] loading test split (n=all)
[eval] 21935 queries, 21935 corpus entries in 7.6s
[eval] encoding test corpus with checkpoint model
[eval] encoded in 61.6s
[eval] R@1=0.9261 R@5=0.9823 R@10=0.9854 MRR=0.9522
[eval] per-query → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-5000_full.jsonl
[eval] summary  → /content/drive/MyDrive/adaptmem-bench/ft-code/eval_ft-code-5000_full.summary.json
